In [ ]:
# Step 1: match_sop(There was one failure reported between 2020-05-23 01:00:00 and 2020-05-23 01:30:00 (UTC+8). Unknown root-cause component
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.42)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.41)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.37)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 2: generate_sop_code for 'RCA-Agent Metric Fault Identification SOP'
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
comp_obs = component_analyze(start_time, end_time)
answer = metric_obs + '\n' + comp_obs

```
Generated code for SOP 'RCA-Agent Metric Fault Identification SOP':
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
comp_obs = component_analyze(start_time, end_time)
answer = metric_obs + '\n' + comp_obs
```

In [ ]:
# Step 3: run_sop
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
comp_obs = component_analyze(start_time, end_time)
answer = metric_obs + '\n' + comp_obs

```
[ANOMALY] db_003 LFParaWrite_Per_Sec: rise to 6.15 at 2020-05-23 01:01:00 (median 1.34, thr 3.002, sev 55.9, n=1)
[ANOMALY] db_007 DbTime: rise to 31.5 at 2020-05-23 01:08:00 (median 7.7, thr 21.03, sev 32.1, n=2)
[ANOMALY] db_003 Exec_Per_Sec: rise to 470.1 at 2020-05-23 01:09:00 (median 118.2, thr 449, sev 25.9, n=4)
[ANOMALY] db_007 DFParaWrite_Per_Sec: drop to 0.4 at 2020-05-23 01:07:00 (median 1.315, thr 0.651, sev 1.2, n=2)
Component anomaly overview (ranked):
os_021  metric_anoms=15 (top: Disk_svctm sev=77.8)  trace_anoms=9  log_anoms=0
os_001  metric_anoms=17 (top: CPU_iowait_time sev=41.6)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=13 (top: Disk_await sev=49.3)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=13 (top: Processor_load_15_min sev=42.7)  trace_anoms=0  log_anoms=0
docker_002  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=13  log_anoms=0
os_018  metric_anoms=12 (top: Disk_rd_kbs sev=77.9)  trace_anoms=0  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
os_020  metric_anoms=11 (top: CPU_iowait_time sev=21.3)  trace_anoms=0  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=11  log_anoms=0
docker_003  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=11  log_anoms=0
db_007  metric_anoms=7 (top: Redo_Per_Sec sev=71.3)  trace_anoms=0  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_008  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=6  log_anoms=0
db_003  metric_anoms=5 (top: Redo_Per_Sec sev=78.1)  trace_anoms=0  log_anoms=0
docker_006  metric_anoms=1 (top: container_cpu_used sev=48.2)  trace_anoms=4  log_anoms=0
```

In [ ]:
# Step 4: match_observation([ANOMALY] db_003 LFParaWrite_Per_Sec: rise to 6.15 at 2020-05-23 01:01:00 (median 1.34, thr 3.002, sev 55.9, n=1)
[ANOMA
...[truncated])

```
ObAgent: The fault class is likely: CPU fault.

Analysis: Multiple hosts show elevated CPU iowait (os_001, os_020) and severe disk service/await anomalies (os_021, os_017, os_018) while DB response time (db_007 DbTime) and write-related DB metrics change. This pattern indicates CPU stalling on I/O (high iowait) impacting DB latency rather than network loss/delay or connection-limit/close issues.
JudgeAgent: No - The history only gives a likely fault class (CPU fault) and multiple anomalous components/metrics (e.g., CPU_iowait_time on os_001/os_020, Disk_svctm on os_021, db_007 DbTime at 2020-05-23 01:08:00, db_003 LFParaWrite_Per_Sec at 2020-05-23 01:01:00) but does not pin a single component from the candidate list, a single anomaly reason (specific KPI tied to that one component), nor a single occurrence datetime for the root cause. Therefore the root cause is not identified correctly/completely.
```

In [ ]:
# Step 5: match_sop(Hypothesized CPU fault / I/O wait and disk service issues based on observation: elevated CPU_iowait on os_001 and os_020
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.33) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.30)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.29)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 6: generate_sop_code for 'RCA-Agent Trace Localization SOP'
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs

```
Generated code for SOP 'RCA-Agent Trace Localization SOP':
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs
```

In [ ]:
# Step 7: run_sop
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
trace_obs = collect_trace(start_time, end_time, None)
metric_obs = whether_is_abnormal_metric(start_time, end_time, '', None)
answer = trace_obs + '\n' + metric_obs

```
[TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0091 at 2020-05-23 01:00:00 (median 0.003, sev 55.3)
[TRACE] docker_003 trace.self.jdbc.duration_p95: rise to 0.007 at 2020-05-23 01:00:00 (median 0.003, sev 53.5)
[TRACE] docker_002 trace.self.jdbc.duration_p95: rise to 0.01 at 2020-05-23 01:09:00 (median 0.003, sev 49.1)
[TRACE] os_021 trace.from_root.osb.error_rate: rise to 0.1087 at 2020-05-23 01:19:00 (median 0, sev 47.6)
[TRACE] docker_001 trace.self.jdbc.duration_p95: rise to 0.017 at 2020-05-23 01:00:00 (median 0.003, sev 47.6)
[TRACE] os_021 trace.self.csf.duration_p95: rise to 39.71 at 2020-05-23 01:18:00 (median 1.555, sev 41.9)
[TRACE] os_021 trace.from_root.osb.duration_mean: rise to 12.2 at 2020-05-23 01:19:00 (median 0.49, sev 38.8)
[TRACE] os_021 trace.self.csf.duration_mean: rise to 6.515 at 2020-05-23 01:18:00 (median 0.4631, sev 38.8)
[TRACE] os_021 trace.from_root.osb.duration_p95: rise to 42.48 at 2020-05-23 01:19:00 (median 1.574, sev 35.4)
[TRACE] docker_004 trace.self.flyremote.duration_p95: rise to 0.093 at 2020-05-23 01:00:00 (median 0.01, sev 28.0)
[TRACE] docker_008 trace.self.local.duration_p95: rise to 0.01 at 2020-05-23 01:00:00 (median 0.009, sev 26.5)
[TRACE] docker_007 trace.self.local.row_count: drop to 128 at 2020-05-23 01:19:00 (median 857.5, sev 24.6)
[TRACE] docker_007 trace.self.jdbc.row_count: drop to 213 at 2020-05-23 01:19:00 (median 1428, sev 24.1)
[TRACE] docker_007 trace.self.local.duration_p95: drop to 0.008 at 2020-05-23 01:18:00 (median 0.009, sev 22.1)
[TRACE] docker_004 trace.self.csf.duration_p95: rise to 0.192 at 2020-05-23 01:12:00 (median 0.1, sev 20.7)
Most-downstream faulty component(s): docker_001, docker_002, docker_003, docker_004, docker_005, docker_006, docker_007, docker_008, os_021, os_022
[ANOMALY] db_003 LFParaWrite_Per_Sec: rise to 6.15 at 2020-05-23 01:01:00 (median 1.34, thr 3.002, sev 55.9, n=1)
[ANOMALY] db_007 DbTime: rise to 31.5 at 2020-05-23 01:08:00 (median 7.7, thr 21.03, sev 32.1, n=2)
[ANOMALY] db_003 Exec_Per_Sec: rise to 470.1 at 2020-05-23 01:09:00 (median 118.2, thr 449, sev 25.9, n=4)
[ANOMALY] db_007 DFParaWrite_Per_Sec: drop to 0.4 at 2020-05-23 01:07:00 (median 1.315, thr 0.651, sev 1.2, n=2)
```

In [ ]:
# Step 8: match_observation([TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0091 at 2020-05-23 01:00:00 (median 0.003, sev 55.3)
[TRACE] 
...[truncated])

```
ObAgent: uncertain

Reason: multiple services show elevated JDBC p95 and DB DbTime rises (slow DB responses), plus strong OS disk-service-time anomalies on os_021. This points to backend I/O-related slowdown rather than an obvious network loss or connection-close pattern, but the available fault classes don't include disk/I/O.

To disambiguate, check:
- whether_is_abnormal_metric(db_007, "ActiveConnections") and whether_is_abnormal_metric(db_007, "ConnectionErrors/ConnFailRate") (to detect connection-limit/close).
- whether_is_abnormal_metric(os_021, "Disk_svctm") and ("Disk_await"/"Disk_util") (to confirm disk I/O bottleneck).
- whether_is_abnormal_metric(os_021, "tcp_retrans_segs" or interface error counters) (to rule in/out network loss/delay).
- collect_trace on the JDBC → DB call chain to measure server-side DB wait (I/O vs network vs connection failure).
JudgeAgent: No - The history only yields a probable fault class (CPU fault / I/O wait) and multiple anomalous components (os_001, os_020, os_021, os_017, os_018, db_007, db_003, etc.) and metrics (CPU_iowait_time, Disk_svctm, DbTime), but it does not pin exactly one component from the candidate list, nor a single anomaly reason tied to that one component, nor a single occurrence minute. Therefore the root cause is not identified correctly/completely.
```

In [ ]:
# Step 9: match_sop(Disk I/O / DB latency / DB connection hypothesis: high Disk_svctm/Disk_await on os_021/os_017/os_018 and elevated CPU_io
...[truncated])

```
Matched SOPs:
- RCA-Agent Metric Fault Identification SOP (score 0.27) (already executed)
  Name: RCA-Agent Metric Fault Identification SOP
Steps:
1. whether_is_abnormal_metric(start_time, end_time, '', None): scan all candidate components and KPIs for compact anomaly evidence and onset times.
2. component_analyze(start_time, end_time): compare the metric evidence with trace/log anomaly counts for the same window.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Trace Localization SOP (score 0.27) (already executed)
  Name: RCA-Agent Trace Localization SOP
Steps:
1. collect_trace(start_time, end_time): summarize abnormal call duration, error-rate, traffic-drop, and downstream information in the incident window.
2. whether_is_abnormal_metric(start_time, end_time, '', None): confirm which trace-localized components are also faulty in metrics.
3. The answer is the observations obtained from the former steps.
- RCA-Agent Level and Root Selection SOP (score 0.26)
  Name: RCA-Agent Level and Root Selection SOP
Steps:
1. component_analyze(start_time, end_time): identify the strongest faulty candidates and their telemetry sources.
2. collect_trace(start_time, end_time): separate faulty upstream/downstream candidates when trace telemetry is available.
3. get_logs(None, start_time, end_time): check whether logs support the same component and reason as the metric/trace evidence.
4. The answer is the observations obtained from the former steps.
```

In [ ]:
# Step 10: generate_sop_code for 'RCA-Agent Level and Root Selection SOP'
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
comp_overview = component_analyze(start_time, end_time)
trace_obs = collect_trace(start_time, end_time, None)
logs_obs = get_logs(None, start_time, end_time)
answer = comp_overview + '\n' + trace_obs + '\n' + logs_obs

```
Generated code for SOP 'RCA-Agent Level and Root Selection SOP':
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
comp_overview = component_analyze(start_time, end_time)
trace_obs = collect_trace(start_time, end_time, None)
logs_obs = get_logs(None, start_time, end_time)
answer = comp_overview + '\n' + trace_obs + '\n' + logs_obs
```

In [ ]:
# Step 11: run_sop
start_time = '2020-05-23 01:00:00'
end_time = '2020-05-23 01:30:00'
comp_overview = component_analyze(start_time, end_time)
trace_obs = collect_trace(start_time, end_time, None)
logs_obs = get_logs(None, start_time, end_time)
answer = comp_overview + '\n' + trace_obs + '\n' + logs_obs

```
Component anomaly overview (ranked):
os_021  metric_anoms=15 (top: Disk_svctm sev=77.8)  trace_anoms=9  log_anoms=0
os_001  metric_anoms=17 (top: CPU_iowait_time sev=41.6)  trace_anoms=0  log_anoms=0
os_017  metric_anoms=13 (top: Disk_await sev=49.3)  trace_anoms=0  log_anoms=0
os_009  metric_anoms=13 (top: Processor_load_15_min sev=42.7)  trace_anoms=0  log_anoms=0
docker_002  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=13  log_anoms=0
os_018  metric_anoms=12 (top: Disk_rd_kbs sev=77.9)  trace_anoms=0  log_anoms=0
docker_004  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=12  log_anoms=0
os_020  metric_anoms=11 (top: CPU_iowait_time sev=21.3)  trace_anoms=0  log_anoms=0
docker_001  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=11  log_anoms=0
docker_003  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=11  log_anoms=0
db_007  metric_anoms=7 (top: Redo_Per_Sec sev=71.3)  trace_anoms=0  log_anoms=0
docker_005  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=7  log_anoms=0
docker_008  metric_anoms=0 (top: n/a sev=0.0)  trace_anoms=6  log_anoms=0
db_003  metric_anoms=5 (top: Redo_Per_Sec sev=78.1)  trace_anoms=0  log_anoms=0
docker_006  metric_anoms=1 (top: container_cpu_used sev=48.2)  trace_anoms=4  log_anoms=0
[TRACE] docker_004 trace.self.jdbc.duration_p95: rise to 0.0091 at 2020-05-23 01:00:00 (median 0.003, sev 55.3)
[TRACE] docker_003 trace.self.jdbc.duration_p95: rise to 0.007 at 2020-05-23 01:00:00 (median 0.003, sev 53.5)
[TRACE] docker_002 trace.self.jdbc.duration_p95: rise to 0.01 at 2020-05-23 01:09:00 (median 0.003, sev 49.1)
[TRACE] os_021 trace.from_root.osb.error_rate: rise to 0.1087 at 2020-05-23 01:19:00 (median 0, sev 47.6)
[TRACE] docker_001 trace.self.jdbc.duration_p95: rise to 0.017 at 2020-05-23 01:00:00 (median 0.003, sev 47.6)
[TRACE] os_021 trace.self.csf.duration_p95: rise to 39.71 at 2020-05-23 01:18:00 (median 1.555, sev 41.9)
[TRACE] os_021 trace.from_root.osb.duration_mean: rise to 12.2 at 2020-05-23 01:19:00 (median 0.49, sev 38.8)
[TRACE] os_021 trace.self.csf.duration_mean: rise to 6.515 at 2020-05-23 01:18:00 (median 0.4631, sev 38.8)
[TRACE] os_021 trace.from_root.osb.duration_p95: rise to 42.48 at 2020-05-23 01:19:00 (median 1.574, sev 35.4)
[TRACE] docker_004 trace.self.flyremote.duration_p95: rise to 0.093 at 2020-05-23 01:00:00 (median 0.01, sev 28.0)
[TRACE] docker_008 trace.self.local.duration_p95: rise to 0.01 at 2020-05-23 01:00:00 (median 0.009, sev 26.5)
[TRACE] docker_007 trace.self.local.row_count: drop to 128 at 2020-05-23 01:19:00 (median 857.5, sev 24.6)
[TRACE] docker_007 trace.self.jdbc.row_count: drop to 213 at 2020-05-23 01:19:00 (median 1428, sev 24.1)
[TRACE] docker_007 trace.self.local.duration_p95: drop to 0.008 at 2020-05-23 01:18:00 (median 0.009, sev 22.1)
[TRACE] docker_004 trace.self.csf.duration_p95: rise to 0.192 at 2020-05-23 01:12:00 (median 0.1, sev 20.7)
Most-downstream faulty component(s): docker_001, docker_002, docker_003, docker_004, docker_005, docker_006, docker_007, docker_008, os_021, os_022
Log telemetry is not available for this dataset. Rely on metrics (whether_is_abnormal_metric) and traces (collect_trace) instead.
```

In [ ]:
# Step 12: match_observation(Component anomaly overview (ranked):
os_021  metric_anoms=15 (top: Disk_svctm sev=77.8)  trace_anoms=9  log_anoms=0
os_0
...[truncated])

```
ObAgent: uncertain

Analysis: Multiple OS nodes show high Disk_svctm / Disk_await and elevated CPU iowait, DB nodes report high Redo_Per_Sec, and many JDBC/service p95 latencies plus an increased error rate — consistent with a storage I/O bottleneck causing requests to slow and fail. This does not cleanly match the provided fault classes (no pure disk option).

To disambiguate, check:
- whether_is_abnormal_metric for disk IO metrics (disk_util%, disk_avg_queue_len, disk_io_time, disk_wait) on affected hosts.
- whether_is_abnormal_metric for net retransmits/packet drops and RTT to rule out network loss/delay.
- whether_is_abnormal_metric for DB connection_count and DB error logs (connection limit/close).
- collect_trace on the DB JDBC spans to see time spent in network vs kernel/fs vs DB.
JudgeAgent: No - The history never pins exactly one component (multiple candidates: os_001, os_021, os_017, os_018, db_007, db_003, etc.), the fault type is ambiguous (observations point to disk I/O / high iowait vs CPU but also DB latency) and not conclusively one of the allowed types, and no single concrete anomalous KPI tied to one chosen component with a specific occurrence minute was established.
```